# Week 10: Weighted Shortest Paths (Dijkstra & Bellman-Ford)

**Z5007 Programming and Data Structures (MTech DSAI Core) · IIT Madras Zanzibar**

Min-heap Dijkstra routing and Bellman-Ford negative-cycle detection.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z5007/week-10/labs/W10_Shortest_Paths_Dijkstra_BellmanFord.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every algorithm is implemented from first principles in Python and verified with assertions.

## 1. Dijkstra's Algorithm on a Zanzibar Transit Graph

With non-negative travel times/distances, popping the minimum-distance frontier vertex from a binary min-heap guarantees its shortest-path distance is finalized in $O((V + E) \log V)$.

In [1]:
import heapq
import math


def dijkstra(adj, source):
    dist = {u: math.inf for u in adj}
    parent = {u: None for u in adj}
    dist[source] = 0.0
    pq = [(0.0, source)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue
        for v, w in adj[u]:
            if d + w < dist[v]:
                dist[v] = d + w
                parent[v] = u
                heapq.heappush(pq, (dist[v], v))
    return dist, parent


def path_to(parent, target):
    p = []
    while target is not None:
        p.append(target)
        target = parent[target]
    return p[::-1]


zanzibar_roads = {
    "StoneTown": [("Mbweni", 6), ("Fumba", 18), ("Bububu", 9)],
    "Mbweni": [("Bweleo", 11), ("Fumba", 14)],
    "Bububu": [("Mahonda", 16)],
    "Fumba": [("Bweleo", 5)],
    "Bweleo": [("Kizimkazi", 38)],
    "Mahonda": [("Nungwi", 33)],
    "Kizimkazi": [],
    "Nungwi": [],
}

dist, parent = dijkstra(zanzibar_roads, "StoneTown")
for dest in ("Bweleo", "Kizimkazi", "Nungwi"):
    print(f"StoneTown -> {dest:10s}: {dist[dest]:4.0f} km via {' -> '.join(path_to(parent, dest))}")
assert dist["Bweleo"] == 17


StoneTown -> Bweleo    :   17 km via StoneTown -> Mbweni -> Bweleo
StoneTown -> Kizimkazi :   55 km via StoneTown -> Mbweni -> Bweleo -> Kizimkazi
StoneTown -> Nungwi    :   58 km via StoneTown -> Bububu -> Mahonda -> Nungwi


## 2. Bellman-Ford & Negative-Cycle Detection

Relaxing all $E$ edges for $V - 1$ passes handles negative edge weights in $O(VE)$; a $V$-th pass improvement certifies a reachable negative-weight cycle.

In [2]:
def bellman_ford(vertices, edges, source):
    dist = {u: math.inf for u in vertices}
    dist[source] = 0.0
    for _ in range(len(vertices) - 1):
        changed = False
        for u, v, w in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                changed = True
        if not changed:
            break
    neg_cycle = any(dist[u] + w < dist[v] for u, v, w in edges)
    return dist, neg_cycle


d_ok, cyc_ok = bellman_ford(["S", "A", "B", "C"], [("S", "A", 4), ("S", "B", 5), ("B", "A", -3), ("A", "C", 2)], "S")
assert not cyc_ok and d_ok["C"] == 4
_, cyc_bad = bellman_ford(["S", "A", "B"], [("S", "A", 1), ("A", "B", -2), ("B", "A", -1)], "S")
assert cyc_bad
print("Bellman-Ford negative edge & cycle detection verified.")


Bellman-Ford negative edge & cycle detection verified.
